# llama.cpp Server — Test Notebook

Tests the running `llamacpp` container (port `10000`) via its OpenAI-compatible API.

Make sure the server is up:
```bash
docker compose up -d
docker compose logs -f llamacpp
```

In [1]:
%pip install -q openai

/home/alif-pc/Codes/inference-engine-deployment/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:10000/v1", api_key="none")

## Text test

In [17]:
resp = client.chat.completions.create(
    model="qwen",
    messages=[
        {
            "role": "user",
            "content": "Reply with exactly: Hello.",
        }
    ],
    max_tokens=32,
    temperature=0.7,
    top_p=0.8,
    extra_body={
        "chat_template_kwargs": {"enable_thinking": False},
    },
)

msg = resp.choices[0].message
content = (msg.content or "").strip()
reasoning = (getattr(msg, "reasoning_content", "") or "").strip()

print("=== Response ===")
print(content if content else "[EMPTY_CONTENT]")
print(f"content_len={len(content)} reasoning_len={len(reasoning)}")

=== Response ===
Hello.
content_len=6 reasoning_len=0


In [18]:
import time
from openai import OpenAI

client = OpenAI(
    api_key="EMPTY",
    base_url="http://localhost:10000/v1",
    timeout=3600,
)

messages = [
    {
        "role": "system",
        "content": "Answer briefly in one sentence.",
    },
    {
        "role": "user",
        "content": "Why did image input fail earlier?",
    },
]

start = time.time()
response = client.chat.completions.create(
    model="Qwen3.5-27B-Q3_K_M.gguf",
    messages=messages,
    max_tokens=64,
    temperature=0.7,
    top_p=0.8,
    extra_body={
        "chat_template_kwargs": {"enable_thinking": False},
    },
)

msg = response.choices[0].message
content = (msg.content or "").strip()
reasoning = (getattr(msg, "reasoning_content", "") or "").strip()
print(f"Response costs: {time.time() - start:.2f}s")
print(f"Generated text: {content if content else '[EMPTY_CONTENT]'}")
print(f"content_len={len(content)} reasoning_len={len(reasoning)}")

Response costs: 0.93s
Generated text: I cannot determine the specific cause of your earlier image input failure without access to the system logs or error messages from that session.
content_len=144 reasoning_len=0


## LangChain + LangGraph quick test

This section uses your local OpenAI-compatible endpoint at `http://localhost:10000/v1` to evaluate concise responses and latency.

In [19]:
import time
from langchain_openai import ChatOpenAI

lc_model = ChatOpenAI(
    api_key="EMPTY",
    base_url="http://localhost:10000/v1",
    model="Qwen3.5-27B-Q3_K_M.gguf",
    temperature=0.7,
    max_tokens=64,
    model_kwargs={
        "top_p": 0.8,
        "extra_body": {
            "chat_template_kwargs": {"enable_thinking": False},
        },
    },
)

start = time.time()
lc_resp = lc_model.invoke("Reply in <= 8 words: say hello.")
print(f"LangChain latency: {time.time() - start:.2f}s")
print("LangChain output:", (lc_resp.content or "").strip() or "[EMPTY_CONTENT]")

/home/alif-pc/Codes/inference-engine-deployment/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3641: UserWarning: Parameters {'extra_body', 'top_p'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  if await self.run_code(code, result, async_=asy):


LangChain latency: 0.21s
LangChain output: Hello!


In [21]:
import time
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class FlowState(TypedDict):
    question: str
    answer: str


def call_model(state: FlowState):
    t0 = time.time()
    out = lc_model.invoke(state["question"]).content or ""
    print(f"LangGraph node latency: {time.time() - t0:.2f}s")
    return {"answer": out.strip()}


graph_builder = StateGraph(FlowState)
graph_builder.add_node("call_model", call_model)
graph_builder.add_edge(START, "call_model")
graph_builder.add_edge("call_model", END)
graph = graph_builder.compile()

result = graph.invoke({"question": "Reply in <= 8 words: what failed earlier?"})
print("LangGraph output:", result["answer"] or "[EMPTY_CONTENT]")

LangGraph node latency: 0.34s
LangGraph output: The previous attempt failed due to a timeout.
